# Cleaning Downloaded Data from avian-influenza

Author: Alexander Maksiaev

Purpose: Clean downloaded data from Andersen Lab's avian-influenza GitHub, rename sequences according to convention

Notes:
* This file must be in the same folder as "utils.py"
* Before running this code, ensure that fork is updated

## Housekeeping

In [1]:
input("Fork updated?")

''

In [1]:
# Libraries

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "11-01-2021"
end_date = "06-19-2026"
date_range = start_date + "--" + end_date

# Maintenance genotypes
genotypes = ["B3.13"] # , "D1.1", "Not"]

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"

references = home + "references/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
ncbi_complete = downloads + "NCBI_Virus/complete/" + date_range + "_Antarctica_North_America_South_America/"

complete_files = originals + "complete/" + date_range + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
combined_files = downloads + "Combinations/NCBI_Virus_Andersen/" + date_range + "_Antarctica_North_America_South_America_ncbi_first/"
if not os.path.exists(combined_files): # checking if the directory exists or not
    os.makedirs(combined_files) # if the directory is not present then create it
metadata_folder = originals + "avian-influenza/metadata/"

os.chdir(references)
state_ref = pd.read_csv("states_ref.csv")

# # Get list of all genotypes
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])



## Read Metadata 

In [3]:
# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
metadata = metadata[metadata["is_retracted"] == False]

print(len(metadata)) 
# display(metadata[metadata["geo_loc_name"] != "United States///"]) # ["geo_loc_name"])
# display(metadata)

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_95088\3314208339.py:3: DtypeWarning: Columns (16,32,36) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")


22138
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
21767


## De-duplicate from NCBI Virus

In [4]:
# Get NCBI Virus SRA sequences
ncbi_sras = []
for dirpath, dirs, files in os.walk(ncbi_complete):
    for file in files: # 8 * x genotypes
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, state_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            # Some accessions may not be SRA
            for value in sra_accessions.values:
                if "SRR" in value:
                    ncbi_sras.append(value)
            # Some duplicates may only be such because of duplicate isolates
            isolates = fasta_file["Isolate_Id"]
            for value in isolates.values:
                ncbi_sras.append(value)
            # Need partial isolates -- e.g. "012345-001" instead of "25-012345-001-original"
            partials = fasta_file["Partials"]
            for value in partials.values:
                ncbi_sras.append(value)
    break 

metadata["Partials"] = metadata["isolate"].apply(lambda x: partial_isolate(x) if x == x else x)

# Remove duplicates from Andersen
for value in ncbi_sras: # to remove
    if "SRR" in value:
        metadata = metadata[metadata["Run"] != value]
    
    metadata = metadata[metadata["isolate"] != value]
    metadata = metadata[metadata["Partials"] != value]
    # print(value)
    
print(len(metadata))

17400


## Naming convention -- relabeling sequences


>[SRA_Accession]|A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year, collection date, geo location, genotype

We need: host_type

host = Host

geo_loc_name = geo_loc_name

geo_location = country (abbreviated)-geo_loc_name (abbreviated) e.g. USA-MD

isolate = isolate

collection date = Collection_Date

serotype = H5N1, etc.

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

### Get genotype

In [5]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata_unassigned = metadata[metadata["Genotype"].str.contains("Not")]
metadata_genotypes = metadata[metadata["Genotype"].isin(genotypes)] # | metadata["Genotype"].str.contains("Not assigned")]

metadata = pd.concat([metadata_unassigned, metadata_genotypes]) if "Not" in genotypes else metadata_genotypes

print(len(metadata)) 
display(metadata)

792


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,Partials,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
1,SRR28752522,WGS,224.74,57594496,PRJNA1102327,SAMN41019181,Viral,34756155,USDA-NVSL,2024-02-23,...,,006483-001,2025-05-09_10-47-11,SRR28752522.fa,B3.13,"PA:ea1, PB2:am2.2, NA:ea1, HA:ea1, MP:ea1, PB1...","ea1:22-003707-003:PA, am2.2:22-010445-001:PB2,...","99.21%, 98.77%, 99.01%, 98.77%, 99.08%, 99.38%...","17, 28, 14, 21, 9, 14, 5, 7",Ran on FASTA - No Coverage Report
2,SRR28752527,WGS,147.04,52791148,PRJNA1102327,SAMN41019203,Viral,16511544,USDA-NVSL,2024,...,,008749-006,2025-05-09_10-47-11,SRR28752527.fa,B3.13,"HA:ea1, PB2:am2.2, PB1:am4, PA:ea1, MP:ea1, NS...","ea1:22-003707-003:HA, am2.2:22-010445-001:PB2,...","98.71%, 98.90%, 99.52%, 99.21%, 98.78%, 99.28%...","22, 25, 11, 17, 12, 6, 8, 14",Ran on FASTA - No Coverage Report
3,SRR28752554,WGS,146.89,78248011,PRJNA1102327,SAMN41019336,Viral,25035449,USDA-NVSL,2024,...,,009581-002,2025-05-09_10-47-14,SRR28752554.fa,B3.13,"PA:ea1, PB1:am4, NP:am8, NS:am1.1, HA:ea1, PB2...","ea1:22-003707-003:PA, am4:23-001855-001:PB1, a...","99.16%, 99.52%, 99.33%, 99.28%, 98.77%, 98.77%...","18, 11, 10, 6, 21, 28, 12, 11",Ran on FASTA - No Coverage Report
7,SRR28834853,WGS,146.26,41841348,PRJNA1102327,SAMN41100352,Viral,13610493,USDA-NVSL,2024-04-01,...,,010192-003,2025-05-09_10-47-27,SRR28834853.fa,B3.13,"HA:ea1, NS:am1.1, PB1:am4, MP:ea1, PB2:am2.2, ...","ea1:22-003707-003:HA, am1.1:22-010085-001:NS, ...","98.71%, 99.05%, 99.56%, 98.78%, 98.63%, 99.25%...","22, 8, 4, 12, 25, 12, 13, 9",Ran on FASTA - No Coverage Report
14,SRR28834889,WGS,229.58,29793398,PRJNA1102327,SAMN41106832,Viral,17342944,USDA-NVSL,2024-03-21,...,,009495-005,2025-05-09_10-47-28,SRR28834889.fa,B3.13,"MP:ea1, PB2:am2.2, NS:am1.1, NP:am8, PA:ea1, N...","ea1:22-003707-003:MP, am2.2:22-010445-001:PB2,...","98.88%, 99.20%, 99.28%, 99.33%, 99.13%, 99.08%...","11, 5, 6, 10, 5, 13, 12, 3",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17375,SRR39224934,WGS,129.21,40058855,PRJNA1102327,SAMN60995289,Viral,14846931,USDA-NVSL,2026-06,...,,07192-003,2026-06-19_07-07-42,SRR39224934.fa,B3.13,"PB2:am2.2, PA:ea1, MP:ea1, HA:ea1, PB1:am4, NS...","am2.2:22-010445-001:PB2, ea1:22-003707-003:PA,...","98.42%, 98.56%, 98.68%, 99.18%, 99.25%, 98.21%...","36, 31, 13, 14, 17, 15, 19, 15",Ran on FASTA - No Coverage Report
17376,SRR39224935,WGS,125.34,20678560,PRJNA1102327,SAMN60995288,Viral,7616196,USDA-NVSL,2026-06,...,,07191-003,2026-06-19_07-07-42,SRR39224935.fa,B3.13,"HA:ea1, MP:ea1, PA:ea1, NP:am8, NA:ea1, PB2:am...","ea1:24-009110-019:HA, ea1:22-003707-003:MP, ea...","99.06%, 98.82%, 98.60%, 98.66%, 99.00%, 98.50%...","16, 11, 30, 20, 14, 34, 16, 14",Ran on FASTA - No Coverage Report
17377,SRR39224936,WGS,128.22,26305547,PRJNA1102327,SAMN60995287,Viral,9662788,USDA-NVSL,2026-06,...,,07191-002,2026-06-19_07-07-42,SRR39224936.fa,B3.13,"PA:ea1, NP:am8, NS:am1.1, NA:ea1, HA:ea1, MP:e...","ea1:22-003707-003:PA, am8:23-032005-001:NP, am...","98.61%, 98.66%, 98.45%, 99.00%, 99.06%, 98.88%...","30, 20, 13, 14, 16, 11, 34, 16",Ran on FASTA - No Coverage Report
17378,SRR39224937,WGS,129.65,24480605,PRJNA1102327,SAMN60995278,Viral,8968732,USDA-NVSL,2026-06,...,,07176-002,2026-06-19_07-07-42,SRR39224937.fa,B3.13,"PB1:am4, PB2:am2.2, MP:ea1, HA:ea1, PA:ea1, NS...","am4:23-001855-001:PB1, am2.2:22-010445-001:PB2...","99.34%, 98.46%, 98.86%, 99.06%, 98.56%, 98.45%...","15, 35, 11, 16, 31, 13, 20, 15",Ran on FASTA - No Coverage Report


### Get specific geolocation

In [17]:
# Format: USA-[state abbreviation], e.g. USA-MD
def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

    state_ref = pd.read_csv(state_ref_file)

    # Normalize the locations

    # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

    metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

    metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

    # Get country
    metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
                                                                              else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
                                                                              else x)

    # Get state
    metadata["Geo_Location_State"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

    try:    
        metadata["Geo_Location_State"] = metadata["Geo_Location_State"].apply(lambda x: 
        # If "x" is the abbreviated state (e.g. "MD")
        state_ref[state_ref['Abbreviation'].str.contains(x)]['Abbreviation'].values[0] # .iloc[0]
        if state_ref["Abbreviation"].str.contains(x).any()
        # If "x" is the state name (e.g. "Maryland")
        else state_ref[state_ref['State'].str.contains(x)]['Abbreviation'].values[0] # .iloc[0] 
        if state_ref["State"].str.contains(x).any() 
        # If "x" has neither the state abbreviation nor the full state name
        else x)
    except:
        print("Could not find states in reference.")

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    # Log metadata
    return metadata

In [18]:


# Double-check state with genbank_mapping
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

# Merge with genbank_mapping
genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
metadata = metadata.merge(genbank_mapping, how="left")

os.chdir(references)
# Get the name of the state, unless it's not in genbank_mapping -- then get it from normalized metadata
metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x 
                                                                else x)
metadata["name_state_genbank"] = metadata["name_state_genbank"].fillna(metadata["name_state"])

metadata["Geo_Location"] = metadata["name_state_genbank"]
metadata = geo_location_get(metadata=metadata)

# # forbidden_chars = [", ", ": "] # List of characters to replace
# # Format: USA-[state abbreviation], e.g. USA-MD
# metadata["Geo_Location"] = metadata["name_state_genbank"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
#                                                         lambda x: 
#                                                         # If "x" has the state abbreviation (e.g. "MD")
#                                                         state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
#                                                         + "-" + 
#                                                         x.split(" ")[-1]
#                                                         if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
#                                                         # If "x" has the full state name (e.g. "Maryland")
#                                                         else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
#                                                         + "-" + 
#                                                         state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
#                                                         if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
#                                                         # If "x" has neither the state abbreviation nor the full state name nor is "USA"
#                                                         else 
#                                                         x
#                                                         )

# # If USA-, delete -
# metadata["Geo_Location"] = metadata["Geo_Location"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)

# # Rename variable back to metadata as we merge metadata and metadata_genbank
# # metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,seg,genbank_acc,genbank_seg,genbank_name,name_state_genbank,Geo_Location,Geo_Location_Normalized,Geo_Location_Country,Geo_Location_State,Geo_Location_New
0,SRR28752522,WGS,224.74,57594496,PRJNA1102327,SAMN41019181,Viral,34756155,USDA-NVSL,2024-02-23,...,HA,PP755382.1,4.0,A/skunk/New Mexico/24-006483-001/2024,New Mexico,New Mexico,New_Mexico-New_Mexico,USA,NM,USA-NM
1,SRR28752522,WGS,224.74,57594496,PRJNA1102327,SAMN41019181,Viral,34756155,USDA-NVSL,2024-02-23,...,MP,PP755383.1,7.0,A/skunk/New Mexico/24-006483-001/2024,New Mexico,New Mexico,New_Mexico-New_Mexico,USA,NM,USA-NM
2,SRR28752522,WGS,224.74,57594496,PRJNA1102327,SAMN41019181,Viral,34756155,USDA-NVSL,2024-02-23,...,NaN,PP755384.1,6.0,A/skunk/New Mexico/24-006483-001/2024,New Mexico,New Mexico,New_Mexico-New_Mexico,USA,NM,USA-NM
3,SRR28752522,WGS,224.74,57594496,PRJNA1102327,SAMN41019181,Viral,34756155,USDA-NVSL,2024-02-23,...,NP,PP755385.1,5.0,A/skunk/New Mexico/24-006483-001/2024,New Mexico,New Mexico,New_Mexico-New_Mexico,USA,NM,USA-NM
4,SRR28752522,WGS,224.74,57594496,PRJNA1102327,SAMN41019181,Viral,34756155,USDA-NVSL,2024-02-23,...,NS,PP755386.1,8.0,A/skunk/New Mexico/24-006483-001/2024,New Mexico,New Mexico,New_Mexico-New_Mexico,USA,NM,USA-NM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1262,SRR39224934,WGS,129.21,40058855,PRJNA1102327,SAMN60995289,Viral,14846931,USDA-NVSL,2026-06,...,NaN,NaN,NaN,NaN,,,-,USA,AL,USA-AL
1263,SRR39224935,WGS,125.34,20678560,PRJNA1102327,SAMN60995288,Viral,7616196,USDA-NVSL,2026-06,...,NaN,NaN,NaN,NaN,,,-,USA,AL,USA-AL
1264,SRR39224936,WGS,128.22,26305547,PRJNA1102327,SAMN60995287,Viral,9662788,USDA-NVSL,2026-06,...,NaN,NaN,NaN,NaN,,,-,USA,AL,USA-AL
1265,SRR39224937,WGS,129.65,24480605,PRJNA1102327,SAMN60995278,Viral,8968732,USDA-NVSL,2026-06,...,NaN,NaN,NaN,NaN,,,-,USA,AL,USA-AL


### Collection Dates

In [19]:
# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(str(x), default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

0       2024-02-23
1       2024-02-23
2       2024-02-23
3       2024-02-23
4       2024-02-23
           ...    
1262       2026-06
1263       2026-06
1264       2026-06
1265       2026-06
1266       2026-06
Name: Collection_Date, Length: 1267, dtype: object


### Get host type

In [20]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and animal == animal: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
            wild_avian domestic_avian               cattle        feline  \
0     great_horned_owl       pheasant            dairy_cow           cat   
1         common_raven         turkey               cattle  domestic_cat   
2        cooper's_hawk        chicken  cattle milk product     feral_cat   
3         coopers_hawk          goose          bovine_milk        feline   
4              peafowl    guinea_fowl              bovine   domestic-cat   
...                ...            ...                  ...           ...   
1663               NaN            NaN                  NaN           NaN   
1664               NaN            NaN                  NaN           NaN   
1665               NaN            NaN                  NaN           NaN   
1666               NaN            NaN                  NaN           NaN   
1667               NaN            NaN                  NaN           NaN   

       other_mammal               human         other      pet_food  \
0        deer

In [21]:
# Ensure that user checks animal output
input("Check animals output. Afterwards, press ESCAPE to continue.")

''

In [22]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

### Make names using all the attributes we collected

In [24]:
# Make names

# metadata["isolate_name"] = metadata["genbank_name"]

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

metadata["isolate_name"] = np.where(metadata["genbank_name"] == "", "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0]) + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)), metadata["genbank_name"])


names = ">" + metadata["Run"] + "|" + metadata["isolate_name"] + "|" + metadata["serotype"] + "|" + metadata["Geo_Location_New"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x) if "-" not in str(x) else str(dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

# print(set(metadata["serotype"]))

0       >SRR28752522|A/skunk/New Mexico/24-006483-001/...
1       >SRR28752522|A/skunk/New Mexico/24-006483-001/...
2       >SRR28752522|A/skunk/New Mexico/24-006483-001/...
3       >SRR28752522|A/skunk/New Mexico/24-006483-001/...
4       >SRR28752522|A/skunk/New Mexico/24-006483-001/...
                              ...                        
1262    >SRR39224934|A/cattle/United States/26G07192-0...
1263    >SRR39224935|A/cattle/United States/26G07191-0...
1264    >SRR39224936|A/cattle/United States/26G07191-0...
1265    >SRR39224937|A/cattle/United States/26G07176-0...
1266    >SRR39224938|A/cattle/United States/26G07051-0...
Name: Name, Length: 1267, dtype: object

In [25]:
# Drop duplicate runs 

metadata["Partials"] = metadata["isolate"].apply(partial_isolate)
metadata = metadata.drop_duplicates(subset=["Partials", "years"], keep="last") # Isolates may be identical, first=NCBI Virus, last=Andersen

In [26]:
os.chdir(complete_files)

print(metadata)
# Save metadata
metadata.to_csv("Andersen_metadata_" + date_range + ".csv")

              Run Assay Type  AvgSpotLen     Bases    BioProject  \
7     SRR28752522        WGS      224.74  57594496  PRJNA1102327   
8     SRR28752527        WGS      147.04  52791148  PRJNA1102327   
9     SRR28752554        WGS      146.89  78248011  PRJNA1102327   
16    SRR28834853        WGS      146.26  41841348  PRJNA1102327   
24    SRR28834889        WGS      229.58  29793398  PRJNA1102327   
...           ...        ...         ...       ...           ...   
1262  SRR39224934        WGS      129.21  40058855  PRJNA1102327   
1263  SRR39224935        WGS      125.34  20678560  PRJNA1102327   
1264  SRR39224936        WGS      128.22  26305547  PRJNA1102327   
1265  SRR39224937        WGS      129.65  24480605  PRJNA1102327   
1266  SRR39224938        WGS      106.53  37024463  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
7     SAMN41019181          Viral  34756155   USDA-NVSL      2024-02-23  ...   
8     SAMN41019203     

## Make FASTA files

In [27]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

# for segment in segments:
#     pair = "Unassigned_" + segment
#     pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0].split(" ")[0] # Make sure "Not assigned" stays as "Not"

                    # if "Not assigned" in genotype:
                    #     genotype = "Unassigned"
                    # print(header)
                    print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.1

## Concatenate with NCBI Virus

In [28]:
# Create fasta files 
 
os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty

        output_path = complete_files + pair + "_" + date_range + ".fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR28752522|A/skunk/New Mexico/24-006483-001/2024|H5N1|USA-NM|2024-02-23|other_mammal|B3.13
>SRR28752527|A/cattle/United States/24-008749-006-v/2024|H5N1|USA-AL|2024|cattle|B3.13
>SRR28752554|A/cattle/United States/24-009581-002-original/2024|H5N1|USA-AL|2024|cattle|B3.13
>SRR28834853|A/cattle/New Mexico/24-010192-003/2024|H5N1|USA-NM|2024-04-01|cattle|B3.13
>SRR28834889|A/cattle/Texas/24-009495-005/2024|H5N1|USA-TX|2024-03-21|cattle|B3.13
>SRR28834890|A/cattle/Texas/24-009495-004/2024|H5N1|USA-TX|2024-03-21|cattle|B3.13
>SRR28834891|A/cattle/Texas/24-009310-009/2024|H5N1|USA-TX|2024-03-21|cattle|B3.13
>SRR28834895|A/domestic cat/Oklahoma/24-009118-002/2024|H5N1|USA-OK|2024-03-20|feline|B3.13
>SRR29040272|A/domestic_cat/United States/24-012039-002/2024|H5N1|USA-AL|2024|feline|B3.13
>SRR28981031|A/domestic_cat/United States/24-012168-003/2024|H5N1|USA-AL|2024|feline|B3.13
>SRR29182383|A/cattle/TX/24-013270-001-original/2024|H5N1|USA-TX|2024-04-30|cattle|B3.13
>SRR29182385|A/cattle/ID/2

In [29]:
# # Concatenate metadata sheets

# os.chdir(ncbi_complete)
# ncbi_metadata = pd.read_csv("NCBI_Virus_" + date_range + "_metadata.csv")
# ncbi_metadata_csv = ncbi_metadata[["Identifier", "GenBank_Title", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]


# # Clean up column names
# metadata = metadata.rename(columns={"Run":"Identifier",  "genbank_name":"GenBank_Title", "serotype":"Serotype", "Geo_Location":"Geo_Location_Abrv", "years":"Years"})
# metadata["Isolate"] = metadata["isolate_name"].apply(lambda x: x.split("/")[-2])
# metadata_csv = metadata[["Identifier", "GenBank_Title", "Host", "Collection_Date", "Isolate", "Serotype", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]

# os.chdir(combined_files)
# print(ncbi_metadata_csv.columns)
# print(metadata_csv.columns)
# combined_metadata = pd.concat([ncbi_metadata_csv, metadata_csv], ignore_index=True)
# combined_metadata.to_csv("NCBI_Virus_Andersen_" + date_range + "_metadata.csv")

In [30]:
# Concatenate with new NCBI Virus sequences

os.chdir(combined_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

# ncbi_fastas = {} # Results in # of genotypes * # of segments
# # Grab files
# for dirpath, dirs, files in os.walk(ncbi_complete):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#         # print(file_name)
#             segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
#             fasta_file = fasta_df_complete(file_name, state_ref) # Convert fasta file to dataframe
#             ncbi_fastas[segment_genotype] = fasta_file

# for ncbi_fasta_key in ncbi_fastas:
#     for andersen_fasta_key in fasta_files:
#         if ncbi_fasta_key == andersen_fasta_key: # If the genotypes/segments are the same
#             ncbi_fasta = ncbi_fastas[ncbi_fasta_key]
#             andersen_fasta = fasta_files[andersen_fasta_key]
#             print(ncbi_fasta)
#             print(andersen_fasta[0])
#             combined_fasta = pd.concat([ncbi_fasta, andersen_fasta[0]], ignore_index=True).drop_duplicates(subset="Partials", keep="last")
#             combined_fasta = combined_fasta.rename(columns={"Sequence":"sequence"})
#             combined_fasta["full_header"] = combined_fasta["full_header"].fillna(combined_fasta["Header"]).apply(lambda x: ">" + x if ">" not in x else x)
#             print(combined_fasta)
#             print(combined_fasta.columns)
#             df_to_fasta(combined_fasta, gisaid_fasta_key + "_" + date_range + ".fasta", complete_files)

andersen = home + "Andersen/complete/" + date_range + "/"

# NCBI Virus files
filenames_ncbi = []
# for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
for dirpath, dirs, files in os.walk(ncbi_complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

# Andersen files
filenames_andersen = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_andersen.append(file_name)
    break 

print(filenames_andersen)

common_genotypes = set()
# Concatenate the two -- should not have any overlap due to dates and deduplication 
for a_file in filenames_andersen:
    a_file_name = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    print(a_file_name)
    for nv_file in filenames_ncbi:
        nv_file_name = nv_file.split("/")[-1].split("_")[0] + "_" + nv_file.split("/")[-1].split("_")[1]
        print(nv_file_name)
        if a_file_name == nv_file_name: # We have a common genotype
            common_genotypes.add(a_file_name)
            filenames = [a_file, nv_file]
            with open(combined_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
            infile.close()
            outfile.close()

# print(common_genotypes)

for a_file in filenames_andersen:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        print(partial_filename_a)
        for segment in segments:
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile2:
                with open(a_file) as infile2:
                    for line in infile2:
                        outfile2.write(line)
                infile2.close()
                outfile2.close()

for a_file in filenames_ncbi:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        for segment in segments:
            print(partial_filename_a)
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile3:
                with open(a_file) as infile3:
                    for line in infile3:
                        outfile3.write(line)
                infile3.close()
                outfile3.close()

['C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--06-19-2026/Andersen_metadata_11-01-2021--06-19-2026.csv', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--06-19-2026/B3.13_HA_11-01-2021--06-19-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--06-19-2026/B3.13_MP_11-01-2021--06-19-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--06-19-2026/B3.13_NA_11-01-2021--06-19-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--06-19-2026/B3.13_NP_11-01-2021--06-19-2026.fasta', 'C:/Users/maksiaevai.NCBI_N